# Figure 1 — fully reproducible builder (matplotlib)

Este notebook genera una versión **100% reproducible** de la **Figure 1** usando solo Python, `matplotlib`, `numpy`, `pandas`, `PIL` y `scipy`.

## Qué produce
Una figura con cuatro paneles:

- **a**: tres pinturas del mismo estilo (Post-Impressionism), fijadas como baja / intermedia / alta geometría;
- **b**: versiones en luminancia con **iso-luminance contours** legibles;
- **c**: mapas de curvatura multiescala para la pintura intermedia;
- **d**: un esquema limpio de los resúmenes de nivel imagen.

## Selección congelada
- **Low geometry** — Anita Malfatti, *Fernanda de Castro* (1922)
- **Intermediate geometry** — Amrita Sher-Gil, *Tribal Women* (1938)
- **High geometry** — Abraham Manievich, *The Yellow House*

## Insumos mínimos
1. `painting_geometry_phase4_artbench_pilot.zip`  
   (para leer `artbench_pilot_features.csv` y validar los valores del descriptor)
2. Las tres imágenes `.jpg` correspondientes a las tres pinturas  
   (o tener ArtBench extraído localmente)

## Salidas
- `Figure1_multiscale_luminance_geometry_reproducible.png`
- `Figure1_multiscale_luminance_geometry_reproducible.pdf`
- `Figure1_multiscale_luminance_geometry_reproducible.svg`


In [ ]:
# Imports

from pathlib import Path
import io
import zipfile
import math
import textwrap

import numpy as np
import pandas as pd
from PIL import Image

import matplotlib.pyplot as plt
from matplotlib import gridspec
from matplotlib.patches import Rectangle
from scipy.ndimage import gaussian_filter

# Optional Colab upload support
try:
    from google.colab import files
    IN_COLAB = True
except Exception:
    IN_COLAB = False

plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["axes.titlesize"] = 10
plt.rcParams["figure.titlesize"] = 18
plt.rcParams["axes.labelsize"] = 10
plt.rcParams["xtick.labelsize"] = 9
plt.rcParams["ytick.labelsize"] = 9


In [ ]:
# Paths and editable configuration

BASE = Path("/content") if IN_COLAB else Path("/mnt/data")
WORK = BASE / "figure1_reproducible_work"
WORK.mkdir(parents=True, exist_ok=True)

INPUT_ZIP = BASE / "painting_geometry_phase4_artbench_pilot.zip"
ARTBENCH_ROOT = BASE / "artbench_data"   # optional
FIG1_IMAGE_DIR = BASE / "figure1_images"
FIG1_IMAGE_DIR.mkdir(parents=True, exist_ok=True)

OUTDIR = WORK / "output"
OUTDIR.mkdir(parents=True, exist_ok=True)

DESCRIPTOR_COL = "geom__curv__kappa_ref_s2p0_grad_weighted_abs"

FIG1_SELECTION = [
    {
        "role": "Low geometry",
        "style": "post_impressionism",
        "artist_slug": "anita-malfatti",
        "artist_label": "Anita Malfatti",
        "title": "Fernanda de Castro",
        "year": "1922",
        "filename": "anita-malfatti_fernanda-de-castro-1922.jpg",
        "expected_value": 0.252172,
    },
    {
        "role": "Intermediate geometry",
        "style": "post_impressionism",
        "artist_slug": "amrita-sher-gil",
        "artist_label": "Amrita Sher-Gil",
        "title": "Tribal Women",
        "year": "1938",
        "filename": "amrita-sher-gil_tribal-women-1938.jpg",
        "expected_value": 0.324556,
    },
    {
        "role": "High geometry",
        "style": "post_impressionism",
        "artist_slug": "abraham-manievich",
        "artist_label": "Abraham Manievich",
        "title": "The Yellow House",
        "year": "",
        "filename": "abraham-manievich_the-yellow-house.jpg",
        "expected_value": 0.403162,
    },
]

EXPORT_BASENAME = "Figure1_multiscale_luminance_geometry_reproducible"
PNG_DPI = 600


## Subida opcional en Colab

Si estás en Colab y aún no has subido el ZIP ni las imágenes, descomenta la celda siguiente.


In [ ]:
# Optional upload in Colab

# if IN_COLAB:
#     uploaded = files.upload()
#     for name, payload in uploaded.items():
#         target = BASE / name
#         with open(target, "wb") as f:
#             f.write(payload)
#     print("Uploaded:", list(uploaded.keys()))


In [ ]:
# Helper functions

def resize_long_side(pil_img, long_side=256):
    w, h = pil_img.size
    scale = float(long_side) / float(max(w, h))
    new_size = (max(1, round(w * scale)), max(1, round(h * scale)))
    return pil_img.resize(new_size, Image.Resampling.LANCZOS)

def luminance_bt601(arr_rgb):
    arr = np.asarray(arr_rgb, dtype=float)
    y = 0.299 * arr[..., 0] + 0.587 * arr[..., 1] + 0.114 * arr[..., 2]
    y -= y.min()
    denom = y.max() - y.min()
    if denom <= 0:
        return np.zeros_like(y)
    return y / denom

def curvature_dog_scale_normalized(I, sigma_ref, long_side, reference_long_side=512, eps=1e-12, grad_quantile=0.20):
    sigma_px = float(sigma_ref) * float(long_side) / float(reference_long_side)
    kw = dict(sigma=sigma_px, mode="reflect", truncate=3.0)

    Ix  = gaussian_filter(I, order=(0, 1), **kw)
    Iy  = gaussian_filter(I, order=(1, 0), **kw)
    Ixx = gaussian_filter(I, order=(0, 2), **kw)
    Iyy = gaussian_filter(I, order=(2, 0), **kw)
    Ixy = gaussian_filter(I, order=(1, 1), **kw)

    grad2 = Ix * Ix + Iy * Iy
    grad = np.sqrt(grad2)

    kappa = (Ixx * Iy * Iy - 2.0 * Ix * Iy * Ixy + Iyy * Ix * Ix) / np.power(grad2 + eps * eps, 1.5)

    finite = np.isfinite(kappa) & np.isfinite(grad)
    pos = grad[finite & (grad > 0)]
    threshold = np.quantile(pos, grad_quantile) if pos.size else 0.0
    valid = finite & (grad >= threshold)

    return sigma_px * kappa, grad, valid

def panel_letter(ax, s):
    ax.text(
        -0.06, 1.04, s,
        transform=ax.transAxes,
        ha="left", va="bottom",
        fontsize=20, fontweight="bold"
    )

def read_csv_from_phase4_zip(zip_path, inner_name="artbench_pilot_features.csv"):
    with zipfile.ZipFile(zip_path, "r") as zf:
        matches = [n for n in zf.namelist() if n.endswith(inner_name)]
        if not matches:
            raise FileNotFoundError(f"{inner_name} not found inside {zip_path}")
        with zf.open(matches[0]) as f:
            return pd.read_csv(f)

def resolve_image_path(filename):
    p = FIG1_IMAGE_DIR / filename
    if p.exists():
        return p
    p2 = BASE / filename
    if p2.exists():
        return p2
    if ARTBENCH_ROOT.exists():
        hits = list(ARTBENCH_ROOT.rglob(filename))
        if len(hits) == 1:
            return hits[0]
        if len(hits) > 1:
            return hits[0]
    raise FileNotFoundError(
        f"Could not find {filename}. Upload it to {FIG1_IMAGE_DIR} or place it under {ARTBENCH_ROOT}."
    )

def clean_axes(ax):
    ax.set_xticks([])
    ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_visible(False)


In [ ]:
# Load Phase-IV features and validate the frozen selection

assert INPUT_ZIP.exists(), f"Missing input ZIP: {INPUT_ZIP}"

phase4_features = read_csv_from_phase4_zip(INPUT_ZIP)
display_cols = ["style", "artist", "filename", DESCRIPTOR_COL]
print("Phase-IV features shape:", phase4_features.shape)
print("Descriptor column:", DESCRIPTOR_COL)

selected_rows = []
for spec in FIG1_SELECTION:
    hit = phase4_features[
        (phase4_features["style"].astype(str) == spec["style"]) &
        (phase4_features["artist"].astype(str) == spec["artist_slug"]) &
        (phase4_features["filename"].astype(str) == spec["filename"])
    ]
    if len(hit) != 1:
        raise RuntimeError(
            f"Expected exactly one row for {spec['filename']} but found {len(hit)}. "
            f"Please inspect artist slug / filename."
        )
    row = hit.iloc[0].to_dict()
    value = float(row[DESCRIPTOR_COL])
    row["expected_value"] = spec["expected_value"]
    row["abs_diff"] = abs(value - spec["expected_value"])
    row["role"] = spec["role"]
    row["artist_label"] = spec["artist_label"]
    row["title"] = spec["title"]
    row["year"] = spec["year"]
    selected_rows.append(row)

selected_df = pd.DataFrame(selected_rows)
print(selected_df[["role", "artist_label", "title", "filename", DESCRIPTOR_COL, "expected_value", "abs_diff"]])


## Cargar las tres imágenes


In [ ]:
# Load and preprocess the three paintings

records = []
for spec in FIG1_SELECTION:
    img_path = resolve_image_path(spec["filename"])
    pil = Image.open(img_path).convert("RGB")
    pil = resize_long_side(pil, long_side=256)
    rgb = np.asarray(pil)
    lum = luminance_bt601(rgb)

    records.append({
        **spec,
        "path": str(img_path),
        "pil": pil,
        "rgb": rgb,
        "lum": lum,
    })

for r in records:
    print(r["role"], "->", r["path"], "| shape:", r["rgb"].shape)


In [ ]:
# Compute curvature maps for the intermediate painting (Tribal Women)

mid = [r for r in records if r["role"].startswith("Intermediate")][0]
mid_lum = mid["lum"]
long_side = max(mid_lum.shape)

sigma_refs = [1, 2, 4, 8]
curv_maps = {}
valid_masks = {}
clips = {}

for s in sigma_refs:
    kappa, grad, valid = curvature_dog_scale_normalized(
        mid_lum,
        sigma_ref=s,
        long_side=long_side,
        reference_long_side=512,
        grad_quantile=0.20,
    )
    curv_maps[s] = kappa
    valid_masks[s] = valid
    vals = np.abs(kappa[valid])
    clips[s] = float(np.percentile(vals, 99.0)) if vals.size else float(np.nanpercentile(np.abs(kappa), 99.0))

print("Computed curvature maps for sigma_ref =", sigma_refs)


## Generar la figura reproducible


In [ ]:
# Build Figure 1

fig = plt.figure(figsize=(14.2, 10.0), constrained_layout=False)
gs = gridspec.GridSpec(
    3, 12, figure=fig,
    height_ratios=[1.05, 0.98, 1.12],
    hspace=0.34, wspace=0.18
)

fig.suptitle(
    "Figure 1. From painting to multiscale luminance geometry",
    x=0.03, y=0.985, ha="left", va="top",
    fontsize=19, fontweight="bold"
)
fig.text(
    0.03, 0.949,
    "Within one style category, paintings occupy markedly different positions in the level-set geometry measured at an intermediate spatial scale.",
    ha="left", fontsize=11, color="dimgray"
)

for j, r in enumerate(records):
    ax = fig.add_subplot(gs[0, 4*j:4*(j+1)])
    if j == 0:
        panel_letter(ax, "a")
    ax.imshow(r["rgb"])
    clean_axes(ax)
    year = f" ({r['year']})" if r["year"] else ""
    title1 = r["role"]
    title2 = f"{r['artist_label']}, {r['title']}{year}"
    validated = selected_df[selected_df["filename"] == r["filename"]][DESCRIPTOR_COL].iloc[0]
    title3 = rf"$G_{{\sigma=2}} = {validated:.3f}$"
    ax.set_title(title1 + "\n" + title2 + "\n" + title3, pad=6, fontsize=10)

for j, r in enumerate(records):
    ax = fig.add_subplot(gs[1, 4*j:4*(j+1)])
    if j == 0:
        panel_letter(ax, "b")
    lum = r["lum"]
    ax.imshow(lum, cmap="gray", vmin=0, vmax=1)
    levels = [0.15, 0.30, 0.45, 0.60, 0.75, 0.90]
    ax.contour(lum, levels=levels, colors="white", linewidths=0.7, alpha=0.9)
    clean_axes(ax)
    role_word = r["role"].replace(" geometry", "").lower()
    ax.set_title(f"Iso-luminance contours — {role_word}", pad=4, fontsize=10)

sub_c = gridspec.GridSpecFromSubplotSpec(1, 4, subplot_spec=gs[2, 0:8], wspace=0.08)
axs_c = []
ims = []
for j, s in enumerate(sigma_refs):
    ax = fig.add_subplot(sub_c[0, j])
    if j == 0:
        panel_letter(ax, "c")
    ghost = 0.88 * mid_lum + 0.12
    ghost = np.clip(ghost, 0, 1)
    ax.imshow(ghost, cmap="gray", vmin=0, vmax=1)
    kappa = curv_maps[s]
    valid = valid_masks[s]
    clip = clips[s]
    overlay = np.ma.masked_where(~valid, kappa)
    im = ax.imshow(overlay, cmap="coolwarm", vmin=-clip, vmax=clip, alpha=0.82)
    ims.append(im)
    clean_axes(ax)
    ax.set_title(rf"$\sigma_{{ref}} = {s}$", pad=6, fontsize=11)
    axs_c.append(ax)

left = axs_c[0].get_position().x0
right = axs_c[-1].get_position().x1
bottom = min(a.get_position().y0 for a in axs_c) - 0.05
cax = fig.add_axes([left + 0.03, bottom, (right-left) - 0.06, 0.018])
cb = plt.colorbar(ims[1], cax=cax, orientation="horizontal")
cb.set_ticks([-clips[2], 0.0, clips[2]])
cb.set_ticklabels(["negative curvature", "0", "positive curvature"])
cb.outline.set_linewidth(0.6)

axd = fig.add_subplot(gs[2, 8:12])
panel_letter(axd, "d")
axd.set_axis_off()
axd.text(0.02, 0.95, "Image-level summaries", fontsize=14, fontweight="bold", ha="left", va="top")
y0 = 0.83
line_gap = 0.125
entries = [
    r"median($|\tilde{\kappa}|$), mean($|\tilde{\kappa}|$)",
    r"$Q_{75}(|\tilde{\kappa}|)$, $Q_{90}(|\tilde{\kappa}|)$, $Q_{95}(|\tilde{\kappa}|)$",
    r"median($\tilde{\kappa}$), positive fraction $P(\tilde{\kappa}>0)$",
    r"$H(\tilde{\kappa})$ (entropy of signed curvature)",
    r"mean($|\tilde{\kappa}|\,\|\nabla L\|$) (gradient-weighted $|\tilde{\kappa}|$)",
]
for i, txt in enumerate(entries):
    y = y0 - i * line_gap
    axd.add_patch(Rectangle((0.02, y-0.065), 0.95, 0.09, fill=False, linewidth=0.8, edgecolor="0.7", transform=axd.transAxes))
    axd.text(0.05, y-0.02, txt, ha="left", va="center", fontsize=10.5)

axd.add_patch(Rectangle((0.02, 0.05), 0.95, 0.12, fill=True, facecolor="#eaf2fb", edgecolor="#5b8cc0", linewidth=1.2, transform=axd.transAxes))
axd.text(
    0.05, 0.11,
    r"$\Rightarrow$ 10 summaries per scale $\times$ 4 scales = 40 curvature features",
    ha="left", va="center", fontsize=11.5, color="#1f4e79", fontweight="bold"
)

png_path = OUTDIR / f"{EXPORT_BASENAME}.png"
pdf_path = OUTDIR / f"{EXPORT_BASENAME}.pdf"
svg_path = OUTDIR / f"{EXPORT_BASENAME}.svg"
fig.savefig(png_path, dpi=PNG_DPI, bbox_inches="tight")
fig.savefig(pdf_path, bbox_inches="tight")
fig.savefig(svg_path, bbox_inches="tight")
plt.show()

print("Saved:")
print(" -", png_path)
print(" -", pdf_path)
print(" -", svg_path)


## Notas finales

- Esta figura es **reproducible**: mismo código, mismas imágenes, mismo ZIP de Phase IV.
- Si quieres una variante con otro estilo o con otras obras, basta cambiar `FIG1_SELECTION`.
- Si deseas que la figura se vea **más cercana a la versión estética generada por IA**, podemos hacer una versión **v2 reproducible** refinando:
  - tamaño y espaciado;
  - densidad de contours;
  - alpha del overlay de curvatura;
  - tipografía y proporciones del panel d.
